In [1]:
!pip install torch torchvision torchaudio --upgrade
!pip install transformers --upgrade
!pip install sentencepiece

Defaulting to user installation because normal site-packages is not writeable
  Using cached torch-2.11.0-cp313-cp313-win_amd64.whl.metadata (29 kB)
  Using cached torchvision-0.26.0-cp313-cp313-win_amd64.whl.metadata (5.5 kB)
Using cached torch-2.11.0-cp313-cp313-win_amd64.whl (114.6 MB)
Using cached torchvision-0.26.0-cp313-cp313-win_amd64.whl (4.3 MB)

   ---------------------------------------- 0/3 [torchaudio]
   ---------------------------------------- 0/3 [torchaudio]
   ------------- -------------------------- 1/3 [torch]
   ------------- -------------------------- 1/3 [torch]
   ------------- -------------------------- 1/3 [torch]
   ------------- -------------------------- 1/3 [torch]
   ------------- -------------------------- 1/3 [torch]
   ------------- -------------------------- 1/3 [torch]
   ------------- -------------------------- 1/3 [torch]
   ------------- -------------------------- 1/3 [torch]
   ------------- -------------------------- 1/3 [torch]
   -------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


Defaulting to user installation because normal site-packages is not writeable
  Using cached regex-2026.4.4-cp313-cp313-win_amd64.whl.metadata (41 kB)
   ---------------------------------------- 0.0/10.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.4 MB ? eta -:--:--
   - -------------------------------------- 0.3/10.4 MB ? eta -:--:--
   --- ------------------------------------ 0.8/10.4 MB 1.8 MB/s eta 0:00:06
   ---- ----------------------------------- 1.0/10.4 MB 1.6 MB/s eta 0:00:06
   ----- ---------------------------------- 1.3/10.4 MB 1.8 MB/s eta 0:00:06
   ------ --------------------------------- 1.6/10.4 MB 1.6 MB/s eta 0:00:06
   -------- ------------------------------- 2.1/10.4 MB 1.7 MB/s eta 0:00:05
   --------- ------------------------------ 2.4/10.4 MB 1.7 MB/s eta 0:00:05
   ----------- ---------------------------- 2.9/10.4 MB 1.7 MB/s eta 0:00:05
   ------------ --------------------------- 3.1/10.4 MB 1.8 MB/s eta 0:00:05
   --------------- ----

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ------------------- -------------------- 0.5/1.1 MB 3.1 MB/s eta 0:00:01
   ---------------------------------------- 1.1/1.1 MB 2.7 MB/s  0:00:00


In [2]:
import pandas as pd
import nltk
from textblob import TextBlob
from nltk.sentiment import SentimentIntensityAnalyzer
from transformers import pipeline

# Download VADER lexicon
nltk.download('vader_lexicon')

# Load your dataset
df = pd.read_csv("iphone15_reviews.csv")

# Columns available:
# rating, title, review_text

print(df.columns)

# Remove null values from review_text
df = df.dropna(subset=['review_text'])

# -----------------------------------
# TEXTBLOB
# -----------------------------------

def textblob_analysis(text):
    polarity = TextBlob(str(text)).sentiment.polarity

    if polarity > 0:
        label = "Positive"
    elif polarity < 0:
        label = "Negative"
    else:
        label = "Neutral"

    return pd.Series([label, polarity])

df[['TextBlob_Label', 'TextBlob_Score']] = df['review_text'].apply(textblob_analysis)

# -----------------------------------
# VADER
# -----------------------------------

sia = SentimentIntensityAnalyzer()

def vader_analysis(text):
    compound = sia.polarity_scores(str(text))['compound']

    if compound >= 0.05:
        label = "Positive"
    elif compound <= -0.05:
        label = "Negative"
    else:
        label = "Neutral"

    return pd.Series([label, compound])

df[['VADER_Label', 'VADER_Score']] = df['review_text'].apply(vader_analysis)

# -----------------------------------
# BERT
# -----------------------------------

# If torch error comes, run:
# !pip install torch torchvision torchaudio
# !pip install transformers

bert_model = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

def bert_analysis(text):
    result = bert_model(str(text[:512]))[0]

    label = result['label']
    score = result['score']

    return pd.Series([label, score])

df[['BERT_Label', 'BERT_Score']] = df['review_text'].apply(bert_analysis)

# -----------------------------------
# SAVE FINAL FILE
# -----------------------------------

df.to_csv("iphone15_sentiment_final.csv", index=False)

print("Done! File saved as iphone15_sentiment_final.csv")
print(df.head())

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


Index(['rating', 'title', 'review_text'], dtype='object')


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Done! File saved as iphone15_sentiment_final.csv
   rating              title  \
0       5            Awesome   
1       5          Fabulous!   
2       5          Wonderful   
3       5          Fabulous!   
4       5  Worth every penny   

                                         review_text TextBlob_Label  \
0  Switch from OnePlus to iPhone I am stunned wit...       Positive   
1  So beautiful, so elegant, just a vowww Akshay ...       Positive   
2  Amezing camera and all over best phone Neeraj ...       Positive   
3  Awesome Thakur Surya Pratap Singh , Hanumana G...       Positive   
4  Awesome photography experience. Battery backup...       Positive   

   TextBlob_Score VADER_Label  VADER_Score BERT_Label  BERT_Score  
0        1.000000    Positive       0.5106   POSITIVE    0.999803  
1        0.675000    Positive       0.8595   POSITIVE    0.999748  
2        1.000000    Positive       0.6369   NEGATIVE    0.682773  
3        1.000000    Positive       0.6249   POSITIVE    0.

In [3]:
import pandas as pd
import nltk
from textblob import TextBlob
from nltk.sentiment import SentimentIntensityAnalyzer
from transformers import pipeline

# Download VADER lexicon
nltk.download('vader_lexicon')

# Load iPhone 16 dataset
df = pd.read_csv("iphone16_reviews.csv")

# Check columns
print(df.columns)

# Remove null values from review_text
df = df.dropna(subset=['review_text'])

# -----------------------------------
# TEXTBLOB
# -----------------------------------

def textblob_analysis(text):
    polarity = TextBlob(str(text)).sentiment.polarity

    if polarity > 0:
        label = "Positive"
    elif polarity < 0:
        label = "Negative"
    else:
        label = "Neutral"

    return pd.Series([label, polarity])

df[['TextBlob_Label', 'TextBlob_Score']] = df['review_text'].apply(textblob_analysis)

# -----------------------------------
# VADER
# -----------------------------------

sia = SentimentIntensityAnalyzer()

def vader_analysis(text):
    compound = sia.polarity_scores(str(text))['compound']

    if compound >= 0.05:
        label = "Positive"
    elif compound <= -0.05:
        label = "Negative"
    else:
        label = "Neutral"

    return pd.Series([label, compound])

df[['VADER_Label', 'VADER_Score']] = df['review_text'].apply(vader_analysis)

# -----------------------------------
# BERT
# -----------------------------------

# If needed:
# !pip install torch torchvision torchaudio
# !pip install transformers

bert_model = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

def bert_analysis(text):
    result = bert_model(str(text[:512]))[0]

    label = result['label']
    score = result['score']

    return pd.Series([label, score])

df[['BERT_Label', 'BERT_Score']] = df['review_text'].apply(bert_analysis)

# -----------------------------------
# SAVE FINAL FILE
# -----------------------------------

df.to_csv("iphone16_sentiment_final.csv", index=False)

print("Done! File saved as iphone16_sentiment_final.csv")
print(df.head())

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


Index(['rating', 'title', 'review_text', 'date', 'city'], dtype='object')


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Done! File saved as iphone16_sentiment_final.csv
   rating                  title  \
0       4       2,05,719 Ratings   
1       5  Mind-blowing purchase   
2       5               Terrific   
3       5  Mind-blowing purchase   
4       5              Wonderful   

                                         review_text      date  \
0                      6,583 Reviews 1,63,435 24,703       NaN   
1                               Super and cool photo  Mar 2025   
2                           Such an great experience  Oct 2025   
3  with unbeatable video performance. Tip-top per...  Nov 2025   
4                             such a beautiful color  Oct 2025   

                      city TextBlob_Label  TextBlob_Score VADER_Label  \
0                      NaN        Neutral        0.000000     Neutral   
1  Uttara Kannada District       Positive        0.341667    Positive   
2                      NaN       Positive        0.400000    Positive   
3                     Agra       Positive    

In [4]:
import pandas as pd
import nltk
from textblob import TextBlob
from nltk.sentiment import SentimentIntensityAnalyzer
from transformers import pipeline

# Download VADER lexicon
nltk.download('vader_lexicon')

# Load IQOO Z10 dataset
df = pd.read_csv("iqoo_z10_reviews.csv")

# Check columns
print(df.columns)

# Remove null values from review_text
df = df.dropna(subset=['review_text'])

# -----------------------------------
# TEXTBLOB
# -----------------------------------

def textblob_analysis(text):
    polarity = TextBlob(str(text)).sentiment.polarity

    if polarity > 0:
        label = "Positive"
    elif polarity < 0:
        label = "Negative"
    else:
        label = "Neutral"

    return pd.Series([label, polarity])

df[['TextBlob_Label', 'TextBlob_Score']] = df['review_text'].apply(textblob_analysis)

# -----------------------------------
# VADER
# -----------------------------------

sia = SentimentIntensityAnalyzer()

def vader_analysis(text):
    compound = sia.polarity_scores(str(text))['compound']

    if compound >= 0.05:
        label = "Positive"
    elif compound <= -0.05:
        label = "Negative"
    else:
        label = "Neutral"

    return pd.Series([label, compound])

df[['VADER_Label', 'VADER_Score']] = df['review_text'].apply(vader_analysis)

# -----------------------------------
# BERT
# -----------------------------------

bert_model = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

def bert_analysis(text):
    result = bert_model(str(text[:512]))[0]

    label = result['label']
    score = result['score']

    return pd.Series([label, score])

df[['BERT_Label', 'BERT_Score']] = df['review_text'].apply(bert_analysis)

# -----------------------------------
# SAVE FINAL FILE
# -----------------------------------

df.to_csv("iqoo_z10_sentiment_final.csv", index=False)

print("Done! File saved as iqoo_z10_sentiment_final.csv")
print(df.head())

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


Index(['rating', 'title', 'review_text'], dtype='object')


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Done! File saved as iqoo_z10_sentiment_final.csv
   rating                         title  \
0       5  ,093 ratings and 248 reviews   
1       5                Classy product   
2       4               Worth the money   
3       4               Worth the money   
4       5                 Great product   

                                         review_text TextBlob_Label  \
0  1 250 2 94 3 263 4 1,163 5 3,323 59 User revie...       Positive   
1  Product is good and value fore maney and batte...       Positive   
2  Single speaker, its ok for normal vdo watching...       Positive   
3  The product is very nice, really I Love this m...       Positive   
4  This is a nice phone . And its performance is ...       Positive   

   TextBlob_Score VADER_Label  VADER_Score BERT_Label  BERT_Score  
0        0.758333    Positive       0.9657   POSITIVE    0.945311  
1        0.516667     Neutral       0.0258   POSITIVE    0.987901  
2        0.354135    Negative      -0.1323   POSITIVE    0.99